### ADL_Extraction.py

### This script automates the process of ADL data extract and uploading to the MITRE FTP site to FAA.

### Usage:
#####     python adl_faa.py [YYYYMMDD]
#####     If no date is provided, the script defaults to retrieving the report for the previous day.

### Author:
####     Kimia Keyvan

### Date:
####     October 2025

In [ ]:
import sys
import pandas as pd
import numpy as np
import datetime as dt
from datetime import datetime, timedelta
import csv
from typing import List, Tuple
from sqlalchemy import create_engine
import sqlalchemy as sa
import oracledb
import paramiko
import os
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText

In [ ]:
def SendEmail(repDate, status, message):

    sender_email = "...@faa.gov"
    receiver_email = ["..."]
    
    try:
        report_date = f"{repDate[4:6]}/{repDate[6:8]}/{repDate[0:4]}"
    except Exception:
        report_date = repDate

    subject_text = f"{process_name} for {report_date}"
    if status == "success":
        subject_text += " - Success"
    elif status == "error":
        subject_text += " - Failure"
    elif status == "NA":
        subject_text += " - Data not available"
    elif status == "low":
        subject_text += " - Low record counts"
        
    message_text = f"{process_name} for {report_date}\n\n{message}"
    
    email_msg = MIMEMultipart("alternative")
    email_msg["Subject"] = subject_text
    email_msg["From"] = sender_email
    email_msg["To"] = ", ".join(receiver_email)

    message_text_type = MIMEText(message_text, "plain")
    email_msg.attach(message_text_type)

    smtp_server = "..."
    port = 25

    try:
        server = smtplib.SMTP(smtp_server)
        server.ehlo()
        server.sendmail(sender_email, receiver_email, email_msg.as_string())
        print("Email sent.")
    except Exception as e:
        print("Error sending email:", e)
    finally:
        if server:
            server.quit()

In [ ]:
def ConnectToOracle(repDate):
    try:
        #oracledb.init_oracle_client(lib_dir=None) 
        username = "..."
        password = "..."
        host = "..."
        port = "1521"
        service_name = "..."
                
        dsn = f"{host}:{port}/{service_name}"
        connection = oracledb.connect(user=username, password=password, dsn=dsn)

        return connection
    
    except Exception as e:
        error_msg = f"Failed Due to Oracle Connection: {e}"
        SendEmail(repDate,"error",error_msg)  
        sys.exit(1) 

In [ ]:
def ConnectToPostgreSQL(repDate):
    try:
        conn_url = sa.engine.URL.create(
            'postgresql',
            username='...',
            password='...',
            host='...',
            database='...')
        sa.engine.create_engine(conn_url)
                
        engine = create_engine(conn_url)
        connection = engine.connect()

        return connection
    
    except Exception as e:
        error_msg = f"Failed Due to PostgreSQL Connection: {e}"
        SendEmail(repDate,"error",error_msg)  
        sys.exit(1) 

In [ ]:
def ExtractData(repDate):

    MAX_ROWS_PER_FILE = 1_500_000
    OUTPUT_FILE_PREFIX = f"adl_{repDate}"
    OUTPUT_FILE_SUFFIX = ".csv"

    try:
        if DB_Engine == 'O':
            connection = ConnectToOracle(repDate)
            df_results = pd.read_sql(EXEC_QUERY_O, con=connection, params={"run_date_str": repDate})
        else:
            connection = ConnectToPostgreSQL(repDate)
            df_results = pd.read_sql(EXEC_QUERY_P, con=connection, params={"run_date_str": repDate})

        if len(df_results) < low_count_threshold:
            SendEmail(repDate, 'low', f"Row count is less than {low_count_threshold}. Found {len(df_results)} rows.")
            sys.exit(1)

        output_files = []
        total_rows = len(df_results)
        file_index = 1

        # Split dataframe into chunks of MAX_ROWS_PER_FILE
        for start_row in range(0, total_rows, MAX_ROWS_PER_FILE):
            end_row = min(start_row + MAX_ROWS_PER_FILE, total_rows)
            chunk_df = df_results.iloc[start_row:end_row]
            
            output_file = f"{OUTPUT_FILE_PREFIX}_{file_index}{OUTPUT_FILE_SUFFIX}"
            chunk_df.to_csv(output_file, index=False)
            output_files.append(output_file)

            file_index += 1

        return output_files

    except Exception as e:
        SendEmail(repDate, "error", f"Extraction Query Failed: {e}")
        sys.exit(1)

    finally:
        connection.close()


In [ ]:
def ExtractDataLoop(repDate):

    connection = None

    MAX_ROWS_PER_FILE = 1_500_000
    OUTPUT_FILE_PREFIX = f"adl_{repDate}"
    OUTPUT_FILE_SUFFIX = ".csv"

    try:
        if DB_Engine == 'O':
            connection = ConnectToOracle(repDate)
            df_results = pd.read_sql(
                EXEC_QUERY_O,
                con=connection,
                params={"run_date_str": repDate}
            )
        else:
            connection = ConnectToPostgreSQL(repDate)
            df_results = pd.read_sql(
                EXEC_QUERY_P,
                con=connection,
                params={"run_date_str": repDate}
            )

        if len(df_results) < low_count_threshold:
            SendEmail(
                repDate,
                'low',
                f"Row count is less than {low_count_threshold}. Found {len(df_results)} rows. Skipping this day."
            )
            return None

        output_files = []
        total_rows = len(df_results)
        file_index = 1

        for start_row in range(0, total_rows, MAX_ROWS_PER_FILE):
            end_row = min(start_row + MAX_ROWS_PER_FILE, total_rows)
            chunk_df = df_results.iloc[start_row:end_row]

            output_file = f"{OUTPUT_FILE_PREFIX}_{file_index}{OUTPUT_FILE_SUFFIX}"
            chunk_df.to_csv(output_file, index=False)
            output_files.append(output_file)

            file_index += 1

        return output_files

    except Exception as e:
        SendEmail(
            repDate,
            "error",
            f"Extraction Query Failed: {e}. Skipping this day."
        )
        return None

    finally:
        if connection is not None:
            connection.close()

In [ ]:
def SFTPUpload(repDate, local_csv_path):

 

    sftp_server = '...'
    sftp_port = 22
    sftp_username = '...'
    sftp_password = '...'

    client = None
    sftp = None

    # Normalize to list if it's a single path
    if isinstance(local_csv_path, str):
        local_csv_path = [local_csv_path]

    existing_files = [
        f for f in local_csv_path
        if os.path.isfile(f)
    ]

    if not existing_files:
        SendEmail(
            repDate,
            "error",
            "No local CSV files found to upload."
        )
        sys.exit(1)

    try:

        # Create SSH client
        client = paramiko.SSHClient()

        # Automatically trust host key
        client.set_missing_host_key_policy(
            paramiko.AutoAddPolicy()
        )

        # Connect
        client.connect(
            hostname=sftp_server,
            port=sftp_port,
            username=sftp_username,
            password=sftp_password,
            timeout=60,
            banner_timeout=60,
            auth_timeout=60,
            look_for_keys=False,
            allow_agent=False
        )

        # Open SFTP session
        sftp = client.open_sftp()

        # Upload files
        for file_path in existing_files:

            remote_file_name = os.path.basename(file_path)

            remote_path = (
                f"{remote_directory}/{remote_file_name}"
            )

            try:

                print(f"Uploading: {file_path}")

                sftp.put(file_path, remote_path)

                # Verify upload
                sftp.stat(remote_path)

                print(f"Upload successful: {remote_path}")

            except FileNotFoundError:

                error_msg = (
                    f"Remote file not found after "
                    f"transfer: {remote_path}"
                )

                SendEmail(repDate, "error", error_msg)

                sys.exit(1)

            except Exception as e:

                error_msg = (
                    f"SFTP upload failed for "
                    f"{file_path}: {e}"
                )

                SendEmail(repDate, "error", error_msg)

                sys.exit(1)

        # Optional success email
        # SendEmail(
        #     repDate,
        #     "success",
        #     f"Uploaded successfully: {existing_files}"
        # )

    except Exception as e:

        error_msg = (
            f"SFTP connection/setup failed: {e}"
        )

        SendEmail(repDate, "error", error_msg)

        sys.exit(1)

    finally:

        try:
            if sftp is not None:
                sftp.close()
        except:
            pass

        try:
            if client is not None:
                client.close()
        except:
            pass

In [ ]:
def SFTPUpload_old(repDate, local_csv_path):
    sftp_server = '...'
    sftp_username = '...'
    sftp_password = '....'


    # Normalize to list if it's a single path
    if isinstance(local_csv_path, str):
        local_csv_path = [local_csv_path]

    existing_files = [f for f in local_csv_path if os.path.isfile(f)]

    if not existing_files:
        SendEmail(repDate, "error", "No local CSV files found to upload.")
        sys.exit(1)

    try:
        transport = paramiko.Transport((sftp_server, 22))
        transport.connect(username=sftp_username, password=sftp_password)
        sftp = paramiko.SFTPClient.from_transport(transport)

        for file_path in existing_files:
            remote_file_name = os.path.basename(file_path)
            remote_path = f"{remote_directory}/{remote_file_name}"

            try:
                sftp.put(file_path, remote_path)

                # Verify file exists on remote server
                try:
                    sftp.stat(remote_path)
                except FileNotFoundError:
                    SendEmail(repDate, "error", f"Remote file not found after transfer: {remote_path}")
                    #continue
                    sys.exit(1)

            except Exception as e:
                SendEmail(repDate, "error", f"SFTP upload failed for {file_path}: {e}")
                #continue
                sys.exit(1)

        #SendEmail(repDate, 'success', f"Uploaded to FTP successfully: {existing_files}")
        
    except Exception as e:
        SendEmail(repDate, "error", f"SFTP connection/setup failed: {e}")
        sys.exit(1)

    finally:
        try:
            sftp.close()
        except:
            pass
        transport.close()


In [ ]:
def RemoveFiles(repDate, local_paths):

    try:

        if isinstance(local_paths, (str, os.PathLike)):
            local_paths = [str(local_paths)]
        elif not isinstance(local_paths, (list, tuple)):
            local_paths = list(local_paths)
        
        expected_prefix = f"adl_{repDate}_"
        basenames = []
        for p in local_paths:
            bn = os.path.basename(p)
            if bn.startswith(expected_prefix) and bn.endswith(".csv"):
                basenames.append(bn)

        for p in local_paths:
            try:
                if os.path.exists(p):
                    os.remove(p)
        
            except Exception as e:
                SendEmail(repDate, "error", f"Failed to delete local {p}: {e}")    

    
    except Exception as e:
        SendEmail(repDate, "error", f"Failed to delete files: {e}")

In [ ]:
def main():
    argv_len = len(sys.argv)
    num_args = argv_len-1

    if num_args == 0:
        yesterday_d = datetime.now() - timedelta(days=1)
        run_date_str = yesterday_d.strftime('%Y%m%d')
    else:
        run_date_strr = sys.argv[1]

    DB_Engine = 'O' # 'P' for PostgreSQL or 'O' for Oracle
    low_count_threshold = 1
    remote_directory = '...'
    process_name = 'ADL - FAA'
    
    EXEC_QUERY_O = """SELECT * FROM (
                    WITH ADL AS (
                    SELECT FID,NULL AS ADL_ID,DEP_ADL_ID, ARR_ADL_ID,AC, AC_NUMBER, DEST, ACENTR,
                    ORIG, DCENTR, DFIX, EDFT, DP, DTRSN, AFIX, EAFT, STAR, STRSN, USR, TYPE, CTG,
                    CLS, ETD_PREFIX, ETD, ETA_PREFIX, ETA, ARTD, ARTA, SGTD, SGTA, IGTD, IGTA, PGTD,
                    PGTA, PETE, LRTD, LRTA, LGTD, LGTA, ERTD, ERTA, OUT_TIME, OFF_TIME, ON_TIME,
                    IN_TIME, OETD, OETA, BETD, BETA, OCTD, OCTA, CTD, CTA, ASLOT, ASLOT_SUFFIX,
                    CTL_ELEM, CTL_TYPE, CTL_EXMPT, SL_HOLD, DV_REC, UX_CNX, FX_CNX, RZ_CNX, RS_CNX,
                    TO_CNX, DV_CNX, RM_CNX, ALD, GDP, DAS, GSD, TOD, CTL_ALM, CDM_MBR, MAJOR, GCD,
                    LTOD, NRP, LFG, III, ATV, SWP, DVT, ADC, UPT, FCA, WXR, CTL_EVENT_ID, ADL_TIME,
                    DEPARTURE_FLAG, ACID, AFP, SUB, CTOP, CTL_PRGM, NULL AS ENTRY, NULL AS EXIT,
                    NULL AS CR_TIME, NULL AS IENTRY, NULL AS EENTRY, NULL AS OENTRY, NULL AS BENTRY,
                    NULL AS DO, TRUNC(IGTD) AS DATA_DATE, SYSDATE AS LOAD_DATE
                    FROM TEST_ADL.FLIGHTS WHERE
                    IGTD BETWEEN TO_DATE(:run_date_str,'YYYYMMDD') AND TO_DATE(:run_date_str,'YYYYMMDD')+ 1 - 1/86400 AND
                    ADL_TIME BETWEEN TO_DATE(:run_date_str,'YYYYMMDD')-5 AND TO_DATE(:run_date_str,'YYYYMMDD') + 5
                     
                    UNION ALL
                     
                    SELECT FID, ADL_ID,NULL AS DEP_ADL_ID, NULL AS ARR_ADL_ID, AC, AC_NUMBER, DEST,
                    ACENTR, ORIG, DCENTR, DFIX, EDFT, DP, DTRSN, AFIX, EAFT, STAR, STRSN, USR, TYPE,
                    CTG, CLS, ETD_PREFIX, ETD, ETA_PREFIX, ETA, ARTD, ARTA, SGTD, SGTA, IGTD, IGTA,
                    PGTD, PGTA, PETE, LRTD, LRTA, LGTD, LGTA, ERTD, ERTA, OUT_TIME, OFF_TIME, ON_TIME,
                    IN_TIME, OETD, OETA, BETD, BETA, OCTD, OCTA, CTD, CTA, ASLOT, ASLOT_SUFFIX,
                    CTL_ELEM, CTL_TYPE, CTL_EXMPT, SL_HOLD, DV_REC, UX_CNX, FX_CNX, RZ_CNX, RS_CNX,
                    TO_CNX, DV_CNX, RM_CNX, ALD, GDP, DAS, GSD, TOD, CTL_ALM, CDM_MBR, MAJOR, GCD,
                    LTOD, NRP, LFG, III, ATV, SWP, DVT, ADC, UPT, FCA, WXR, CTL_EVENT_ID, ADL_TIME,
                    DEPARTURE_FLAG, ACID, AFP, SUB, CTOP, CTL_PRGM, ENTRY, EXIT, CR_TIME, IENTRY,
                    EENTRY, OENTRY, BENTRY, "do" AS DO, TRUNC(IGTD) AS DATA_DATE, SYSDATE AS LOAD_DATE
                    FROM TEST_ADL.FLOW_AREA_FLIGHTS WHERE
                    IGTD BETWEEN TO_DATE(:run_date_str,'YYYYMMDD') AND TO_DATE(:run_date_str,'YYYYMMDD')+ 1 - 1/86400 AND
                    ADL_TIME BETWEEN TO_DATE(:run_date_str,'YYYYMMDD')-5 AND TO_DATE(:run_date_str,'YYYYMMDD') + 5 )
                     
                    SELECT A.*,E.ELEMENT_NAME,E.ADL_TYPE, E.SUB_FLAG, E.SCS_FLAG,
                    E.ADPT_COMPRESSION_ON_FLAG, E.ADPT_COMPRESSION_IN_ADL_FLAG, E.COMPLETION_FLAG
                    FROM ADL A LEFT JOIN TEST_ADL.PROCESSED_ADLS E ON
                    A.ADL_TIME = E.ADL_TIME AND
                    COALESCE(A.ADL_ID, A.DEP_ADL_ID,A.ARR_ADL_ID) = E.ADL_ID AND
                    E.ADL_TIME BETWEEN TO_DATE(:run_date_str,'YYYYMMDD')-5 AND TO_DATE(:run_date_str,'YYYYMMDD') + 5 )"""
    
    
    
    EXEC_QUERY_P = """WITH ADL AS (
                SELECT FID,NULL AS ADL_ID,DEP_ADL_ID, ARR_ADL_ID,AC, AC_NUMBER, DEST, ACENTR,
                ORIG, DCENTR, DFIX, EDFT, DP, DTRSN, AFIX, EAFT, STAR, STRSN, USR, TYPE, CTG,
                CLS, ETD_PREFIX, ETD, ETA_PREFIX, ETA, ARTD, ARTA, SGTD, SGTA, IGTD, IGTA, PGTD,
                PGTA, PETE, LRTD, LRTA, LGTD, LGTA, ERTD, ERTA, OUT_TIME, OFF_TIME, ON_TIME,
                IN_TIME, OETD, OETA, BETD, BETA, OCTD, OCTA, CTD, CTA, ASLOT, ASLOT_SUFFIX,
                CTL_ELEM, CTL_TYPE, CTL_EXMPT, SL_HOLD, DV_REC, UX_CNX, FX_CNX, RZ_CNX, RS_CNX,
                TO_CNX, DV_CNX, RM_CNX, ALD, GDP, DAS, GSD, TOD, CTL_ALM, CDM_MBR, MAJOR, GCD,
                LTOD, NRP, LFG, III, ATV, SWP, DVT, ADC, UPT, FCA, WXR, CTL_EVENT_ID, ADL_TIME,
                DEPARTURE_FLAG, ACID, AFP, SUB, CTOP, CTL_PRGM, NULL AS ENTRY, NULL AS EXIT,
                NULL AS CR_TIME, NULL AS IENTRY, NULL AS EENTRY, NULL AS OENTRY, NULL AS BENTRY,
                NULL AS DO, DATE_TRUNC('day',IGTD) AS DATA_DATE, CURRENT_TIMESTAMP AT TIME ZONE 'America/New_York' AS LOAD_DATE
                FROM TEST_ADL.FLIGHTS WHERE
                IGTD >= TO_DATE(%(run_date_str)s,'YYYYMMDD')
                AND IGTD < TO_DATE(%(run_date_str)s,'YYYYMMDD') + INTERVAL '1 day' AND
                ADL_TIME BETWEEN TO_DATE(%(run_date_str)s, 'YYYYMMDD') - INTERVAL '5 days' AND TO_DATE(%(run_date_str)s, 'YYYYMMDD') + INTERVAL '5 days'
                 
                UNION ALL
                 
                SELECT FID, ADL_ID,NULL AS DEP_ADL_ID, NULL AS ARR_ADL_ID, AC, AC_NUMBER, DEST,
                ACENTR, ORIG, DCENTR, DFIX, EDFT, DP, DTRSN, AFIX, EAFT, STAR, STRSN, USR, TYPE,
                CTG, CLS, ETD_PREFIX, ETD, ETA_PREFIX, ETA, ARTD, ARTA, SGTD, SGTA, IGTD, IGTA,
                PGTD, PGTA, PETE, LRTD, LRTA, LGTD, LGTA, ERTD, ERTA, OUT_TIME, OFF_TIME, ON_TIME,
                IN_TIME, OETD, OETA, BETD, BETA, OCTD, OCTA, CTD, CTA, ASLOT, ASLOT_SUFFIX,
                CTL_ELEM, CTL_TYPE, CTL_EXMPT, SL_HOLD, DV_REC, UX_CNX, FX_CNX, RZ_CNX, RS_CNX,
                TO_CNX, DV_CNX, RM_CNX, ALD, GDP, DAS, GSD, TOD, CTL_ALM, CDM_MBR, MAJOR, GCD,
                LTOD, NRP, LFG, III, ATV, SWP, DVT, ADC, UPT, FCA, WXR, CTL_EVENT_ID, ADL_TIME,
                DEPARTURE_FLAG, ACID, AFP, SUB, CTOP, CTL_PRGM, ENTRY, EXIT, CR_TIME, IENTRY,
                EENTRY, OENTRY, BENTRY, "do" AS DO, DATE_TRUNC('day',IGTD) AS DATA_DATE, CURRENT_TIMESTAMP AT TIME ZONE 'America/New_York' AS LOAD_DATE
                FROM TEST_ADL.FLOW_AREA_FLIGHTS WHERE
                IGTD >= TO_DATE(%(run_date_str)s,'YYYYMMDD')
                AND IGTD < TO_DATE(%(run_date_str)s,'YYYYMMDD') + INTERVAL '1 day' AND
                ADL_TIME BETWEEN TO_DATE(%(run_date_str)s, 'YYYYMMDD') - INTERVAL '5 days' AND TO_DATE(%(run_date_str)s, 'YYYYMMDD') + INTERVAL '5 days')
                 
                SELECT A.*,E.ELEMENT_NAME,E.ADL_TYPE, E.SUB_FLAG, E.SCS_FLAG,
                E.ADPT_COMPRESSION_ON_FLAG, E.ADPT_COMPRESSION_IN_ADL_FLAG, E.COMPLETION_FLAG
                FROM ADL A LEFT JOIN TEST_ADL.PROCESSED_ADLS E ON
                A.ADL_TIME = E.ADL_TIME AND
                COALESCE(A.ADL_ID, A.DEP_ADL_ID,A.ARR_ADL_ID) = E.ADL_ID AND
                E.ADL_TIME BETWEEN TO_DATE(%(run_date_str)s, 'YYYYMMDD') - INTERVAL '5 days' AND TO_DATE(%(run_date_str)s, 'YYYYMMDD') + INTERVAL '5 days'"""
        
## print the query for validation
#raw_conn = connection.connection
#cursor = raw_conn.cursor()
#print(cursor.mogrify(EXEC_QUERY_P, {"run_date_str": run_date_str}).decode())     
        
    
    try:
        op = ExtractData(run_date_str)

        if op and os.path.isfile(op):
            SFTPUpload(run_date_str, op)
        else:
            SendEmail(run_date_str, 'error',f"CSV Report not found on FTP: {op}")

    except Exception as e:
        SendEmail(run_date_str, "error", f"Code run failed: {e}")
        sys.exit(1)
    
if __name__ == '__main__':
    main()